# Bloque 1: Instalación y Preparación

In [9]:
# Instala la librería si no la tienes
# pip install tensorflow pandas scikit-learn
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Cargamos los datos (asumiendo el mismo CSV de la práctica anterior)
df = pd.read_csv('df_total.csv', encoding='UTF-8')

# Convertimos las etiquetas de texto (ej: "deportes") a números (0, 1, 2...)
le = LabelEncoder()
df['label_num'] = le.fit_transform(df['Type'])
num_clases = len(le.classes_)

# Bloque 2: Tokenización y Padding (El "corazón" del DL en texto)

In [10]:
# Parámetros de configuración
vocab_size = 10000 # Solo usamos las 10,000 palabras más comunes
max_length = 200 # Cortamos o rellenamos las noticias a 200 palabras
trunc_type = 'post'
padding_type = 'post'
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['news'])

# Convertimos texto a secuencias de números
sequences = tokenizer.texts_to_sequences(df['news'])

# Aseguramos que todas midan lo mismo
padded = pad_sequences(sequences, maxlen=max_length,
padding=padding_type, truncating=trunc_type)

# Bloque 3: División de Datos

In [11]:
X_train, X_test, y_train, y_test = train_test_split(padded, df['label_num'], test_size=0.2, random_state=42)

# Bloque 4: Creación de la Red Neuronal

In [12]:
model = tf.keras.Sequential([
# 1. Capa de Embedding: Crea un espacio vectorial para las palabras
tf.keras.layers.Embedding(vocab_size, 16,
input_length=max_length),

# 2. Capa GlobalAveragePooling: Reduce la dimensionalidad
tf.keras.layers.GlobalAveragePooling1D(),

# 3. Capa Densa: Una capa intermedia de neuronas para aprender patrones
tf.keras.layers.Dense(24, activation='relu'),

# 4. Capa de Salida: Una neurona por cada categoría (deportes, política, etc.)
tf.keras.layers.Dense(num_clases, activation='softmax')])

model.compile(loss='sparse_categorical_crossentropy',
optimizer='adam', metrics=['accuracy'])
model.summary()

/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Bloque 5: Entrenamiento

In [13]:
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), verbose=2)

Epoch 1/10
31/31 - 0s - 16ms/step - accuracy: 0.2754 - loss: 1.9182 - val_accuracy: 0.2992 - val_loss: 1.8814
Epoch 2/10
31/31 - 0s - 3ms/step - accuracy: 0.2744 - loss: 1.8443 - val_accuracy: 0.2992 - val_loss: 1.8097
Epoch 3/10
31/31 - 0s - 3ms/step - accuracy: 0.2754 - loss: 1.7760 - val_accuracy: 0.2992 - val_loss: 1.7589
Epoch 4/10
31/31 - 0s - 3ms/step - accuracy: 0.2867 - loss: 1.7277 - val_accuracy: 0.3648 - val_loss: 1.7157
Epoch 5/10
31/31 - 0s - 2ms/step - accuracy: 0.3844 - loss: 1.6735 - val_accuracy: 0.4221 - val_loss: 1.6673
Epoch 6/10
31/31 - 0s - 3ms/step - accuracy: 0.4327 - loss: 1.6118 - val_accuracy: 0.4631 - val_loss: 1.6034
Epoch 7/10
31/31 - 0s - 2ms/step - accuracy: 0.4676 - loss: 1.5326 - val_accuracy: 0.5000 - val_loss: 1.5294
Epoch 8/10
31/31 - 0s - 2ms/step - accuracy: 0.4902 - loss: 1.4442 - val_accuracy: 0.4918 - val_loss: 1.4565
Epoch 9/10
31/31 - 0s - 3ms/step - accuracy: 0.5303 - loss: 1.3539 - val_accuracy: 0.5246 - val_loss: 1.3878
Epoch 10/10
31/31 

# Bloque 6: Evaluación y Predicción

In [14]:
# Evaluamos la precisión final
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Precisión del modelo Deep Learning: {accuracy*100:.2f}%")

# Ejemplo de predicción con una noticia nueva
nueva_noticia = ["El equipo ganó el campeonato de liga en el último minuto"]
secuencia_nueva = tokenizer.texts_to_sequences(nueva_noticia)
padded_nueva = pad_sequences(secuencia_nueva, maxlen=max_length)
prediccion = model.predict(padded_nueva)
clase_predicha = le.inverse_transform([tf.argmax(prediccion[0]).numpy()])

print(f"La noticia se clasifica como: {clase_predicha[0]}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5656 - loss: 1.3238 
Precisión del modelo Deep Learning: 56.56%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
La noticia se clasifica como: Innovacion
